# Mouse Brain Spatial Multiome Model Comparison

This notebook:
1. Loads the Mouse Brain Spatial Multiome dataset (RNA + ATAC with spatial coordinates).
2. Loads all trained topic models under `/data/omics_topic_models/mouse_brain_spatial`.
3. Selects the best model by spatial autocorrelation (Moran's I on clustering assignments).
4. Loads baseline models: SpatialGlue, STAMP, MultiVI, and MOFA+.
5. Compares all models via UMAP and spatial plots colored by Leiden clustering.
6. Evaluates spatial coherence metrics for each embedding.

In [ ]:
import os
import sys
import warnings
from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import mudata as md
import muon as mu
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import scipy.sparse as sp
import squidpy as sq
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings("ignore")

# Data paths
RNA_PATH = "/data/Data_SpatialGlue/Dataset10_Mouse_Brain_H3K27me3/adata_RNA.h5ad"
ATAC_PATH = "/data/Data_SpatialGlue/Dataset10_Mouse_Brain_H3K27me3/adata_peaks_normalized.h5ad"
MODELS_DIR = Path("/data/omics_topic_models/mouse_brain_spatial")
BASELINES_DIR = MODELS_DIR / "baselines"

## 1. Load Mouse Brain Spatial Multiome data

In [ ]:
from omics_topic import MultimodalAmortizedLDA


def ensure_counts_layer(adata):
    if "counts" not in adata.layers:
        adata.layers["counts"] = adata.X.copy()


def binarize_atac(adata_atac):
    X = adata_atac.X
    if sp.issparse(X):
        X = X.tocsr(copy=True)
        X.data = np.ones_like(X.data)
        X.eliminate_zeros()
        adata_atac.layers["binary"] = X
    else:
        adata_atac.layers["binary"] = (X != 0).astype(np.float32)


def load_data():
    """Load and preprocess Mouse Brain Spatial Multiome data."""
    adata_rna = sc.read_h5ad(RNA_PATH)
    adata_atac = sc.read_h5ad(ATAC_PATH)

    # Ensure counts layer for RNA
    ensure_counts_layer(adata_rna)

    # Binarize ATAC data
    binarize_atac(adata_atac)

    # Create MuData
    mdata = md.MuData({"rna": adata_rna, "atac": adata_atac})

    # Build spatial neighbor graph from RNA coordinates
    sc.pp.neighbors(
        mdata.mod["rna"],
        use_rep="spatial",
        n_neighbors=5,
        metric="euclidean",
        key_added="spatial",
    )

    # Share spatial connectivities across modalities
    mdata.obsp["spatial_connectivities"] = mdata.mod["rna"].obsp["spatial_connectivities"]
    mdata.mod["atac"].obsp["spatial_connectivities"] = mdata.mod["rna"].obsp[
        "spatial_connectivities"
    ]
    mdata.mod["atac"].obsp["spatial_distances"] = mdata.mod["rna"].obsp["spatial_distances"]

    # Store spatial coordinates at MuData level
    mdata.obsm["spatial"] = mdata.mod["rna"].obsm["spatial"]

    print(f"RNA: {mdata.mod['rna'].shape}")
    print(f"ATAC: {mdata.mod['atac'].shape}")
    print(f"Total spots: {mdata.n_obs}")

    return mdata


mdata = load_data()
spatial_coords = mdata.obsm["spatial"]

## 2. Load all trained topic models and compute spatial autocorrelation

In [ ]:
def parse_model_config(dirname):
    """Parse hyperparameter configuration from directory name."""
    config = {}
    if "horseshoe" in dirname:
        config["feature_prior_type"] = "horseshoe"
    else:
        config["feature_prior_type"] = "logistic_normal"

    if "weight_cell" in dirname:
        config["weight_mode"] = "cell"
    elif "weight_universal" in dirname:
        config["weight_mode"] = "universal"
    else:
        config["weight_mode"] = "equal"

    if "learnable_disp_global" in dirname:
        config["dispersion"] = "learnable_global"
    elif "learnable_disp_pergene" in dirname:
        config["dispersion"] = "learnable_pergene"
    else:
        config["dispersion"] = "fixed"

    return config


def compute_spatial_autocorrelation(latent, spatial_coords, n_neighbors=6):
    """
    Compute spatial autocorrelation (Moran's I) for Leiden clustering on latent space.
    Returns the mean Moran's I across all Leiden cluster indicator variables.
    """
    # Create AnnData for clustering
    adata = ad.AnnData(np.asarray(latent))
    adata.obsm["spatial"] = np.asarray(spatial_coords)

    # Compute neighbors and Leiden clustering
    sc.pp.neighbors(adata, use_rep="X", n_neighbors=15, metric="cosine")
    sc.tl.leiden(adata, key_added="leiden", resolution=0.5)

    # Build spatial connectivity graph for Moran's I
    sq.gr.spatial_neighbors(adata, n_neighs=n_neighbors, coord_type="generic")

    # Compute Moran's I for each topic dimension
    # Use the latent representation directly
    adata.obs[[f"dim_{i}" for i in range(latent.shape[1])]] = latent
    sq.gr.spatial_autocorr(
        adata,
        mode="moran",
        genes=[f"dim_{i}" for i in range(latent.shape[1])],
        n_perms=100,
    )

    morans_i = adata.uns["moranI"]["I"].mean()
    n_clusters = len(adata.obs["leiden"].unique())

    return {"morans_i": morans_i, "n_clusters": n_clusters, "leiden": adata.obs["leiden"].values}


def compute_topic_spatial_coherence(theta, spatial_coords, n_neighbors=6):
    """
    Compute Moran's I for each topic proportion to measure spatial coherence.
    """
    adata = ad.AnnData(np.asarray(theta))
    adata.obsm["spatial"] = np.asarray(spatial_coords)

    # Build spatial connectivity graph
    sq.gr.spatial_neighbors(adata, n_neighs=n_neighbors, coord_type="generic")

    # Add topic proportions to obs for autocorrelation
    for i in range(theta.shape[1]):
        adata.obs[f"topic_{i}"] = theta[:, i]

    sq.gr.spatial_autocorr(
        adata,
        mode="moran",
        genes=[f"topic_{i}" for i in range(theta.shape[1])],
        n_perms=100,
    )

    return adata.uns["moranI"]["I"].mean()

In [ ]:
# Find all model directories
if not MODELS_DIR.exists():
    print(f"Warning: Models directory does not exist: {MODELS_DIR}")
    print("Creating empty results list. Run training first.")
    model_dirs = []
else:
    model_dirs = sorted(
        [d for d in MODELS_DIR.iterdir() if d.is_dir() and d.name.startswith("prior_")]
    )
print(f"Found {len(model_dirs)} candidate model dirs")

# Prepare data for model loading - IMPORTANT: include spatial_keys for spatial models
if model_dirs:
    mdata_setup, modality_names, feat_counts = MultimodalAmortizedLDA.setup_mudata(
        mdata,
        modality_order=["rna", "atac"],
        layers={"rna": None, "atac": "binary"},
        spatial_keys="spatial_connectivities",  # Required for spatial models with GCN
    )
    adata_flat = mdata.uns["_flattened_ann_data"]
    
    # Verify spatial connectivities are in the flattened AnnData
    if "spatial_connectivities" in adata_flat.obsp:
        print(f"Spatial connectivities available: {adata_flat.obsp['spatial_connectivities'].shape}")
    else:
        print("Warning: spatial_connectivities not found in flattened AnnData")

In [ ]:
results = []

for model_dir in model_dirs:
    model_path = model_dir / "model"
    if not model_path.exists():
        continue

    print(f"Loading {model_dir.name}...")
    
    # First, try to load pre-saved latent representation if available
    # (This is more robust than reloading the full model)
    latent_path = model_dir / "latent_representation.npy"
    theta_path = model_dir / "theta.npy"
    
    theta_array = None
    
    # Check for pre-saved latent representations
    if latent_path.exists():
        theta_array = np.load(latent_path)
        print(f"  Loaded pre-saved latent from {latent_path}")
    elif theta_path.exists():
        theta_array = np.load(theta_path)
        print(f"  Loaded pre-saved theta from {theta_path}")
    
    # If no pre-saved latent, try loading the model
    if theta_array is None:
        try:
            model = MultimodalAmortizedLDA.load(str(model_path), adata=adata_flat)
            theta = model.get_latent_representation(batch_size=mdata.n_obs)
            theta_array = np.asarray(theta)
            
            # Save for future use
            np.save(latent_path, theta_array)
            print(f"  Loaded from model and saved latent to {latent_path}")
        except Exception as e:
            print(f"  Error loading model: {e}")
            # For spatial models with GCN, the load might fail due to architecture mismatch
            # Skip this model if we can't load it
            continue
    
    config = parse_model_config(model_dir.name)

    # Compute metrics (diversity requires model, so we skip if not available)
    try:
        # Spatial autocorrelation
        spatial_metrics = compute_spatial_autocorrelation(theta_array, spatial_coords)
        topic_morans = compute_topic_spatial_coherence(theta_array, spatial_coords)

        result = {
            "model_name": model_dir.name,
            **config,
            "morans_i": spatial_metrics["morans_i"],
            "topic_morans_i": topic_morans,
            "n_clusters": spatial_metrics["n_clusters"],
            "theta": theta_array,
            "leiden": spatial_metrics["leiden"],
        }
        results.append(result)

        print(
            f"  Moran's I: {spatial_metrics['morans_i']:.4f}, "
            f"Topic Moran's I: {topic_morans:.4f}"
        )
    except Exception as e:
        print(f"  Error computing metrics: {e}")
        import traceback
        traceback.print_exc()

print(f"\nSuccessfully loaded {len(results)} models")

## 3. Select best model by spatial autocorrelation

In [ ]:
if results:
    results_display = [{k: v for k, v in r.items() if k not in ["theta", "leiden"]} for r in results]
    df_results = pd.DataFrame(results_display)

    # Sort by Moran's I (spatial autocorrelation) - higher is better
    df_sorted = df_results.sort_values("topic_morans_i", ascending=False)
    print("Model Comparison (sorted by Topic Moran's I):")
    print(df_sorted.to_string(index=False))

    best_idx = df_sorted["topic_morans_i"].idxmax()
    best_model_name = df_sorted.loc[best_idx, "model_name"]
    best_result = next(r for r in results if r["model_name"] == best_model_name)

    print(f"\nBest model: {best_model_name}")
    print(f"  Feature prior: {best_result['feature_prior_type']}")
    print(f"  Weight mode: {best_result['weight_mode']}")
    print(f"  Dispersion: {best_result['dispersion']}")
    print(f"  Topic Moran's I: {best_result['topic_morans_i']:.4f}")
    print(f"  Latent Moran's I: {best_result['morans_i']:.4f}")

    theta_best = best_result["theta"]
else:
    print("No topic models loaded.")
    print("This could be because:")
    print("  1. No models were trained yet (run train_mouse_brain_spatial.py first)")
    print("  2. Trained models don't have latent_representation.npy saved")
    print("     (re-run training with updated script that saves latents)")
    print("  3. Model loading failed due to architecture mismatch")
    theta_best = None
    df_sorted = None

In [ ]:
# Visualize results
if results and len(results) > 1:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    ax = axes[0]
    df_pivot = df_sorted.pivot_table(
        values="topic_morans_i", index="feature_prior_type", columns="weight_mode", aggfunc="mean"
    )
    sns.heatmap(df_pivot, annot=True, fmt=".3f", cmap="viridis", ax=ax)
    ax.set_title("Topic Moran's I by Prior Type and Weight Mode")

    ax = axes[1]
    colors = df_sorted["feature_prior_type"].map({"logistic_normal": "blue", "horseshoe": "red"})
    ax.scatter(df_sorted["morans_i"], df_sorted["topic_morans_i"], c=colors, s=100, alpha=0.7)
    ax.set_xlabel("Latent Moran's I")
    ax.set_ylabel("Topic Moran's I")
    ax.set_title("Latent vs Topic Spatial Autocorrelation")
    ax.legend(
        handles=[
            plt.Line2D([0], [0], marker="o", color="w", markerfacecolor="blue", markersize=10, label="Logistic Normal"),
            plt.Line2D([0], [0], marker="o", color="w", markerfacecolor="red", markersize=10, label="Horseshoe"),
        ]
    )

    plt.tight_layout()
    plt.show()
elif results:
    print("Only one model loaded, skipping comparison visualization")

## 4. Load baseline models

In [ ]:
# Load SpatialGlue latent
latent_spatialglue = None
spatialglue_path = BASELINES_DIR / "latent_spatialglue.npy"
if spatialglue_path.exists():
    latent_spatialglue = np.load(spatialglue_path)
    print(f"Loaded SpatialGlue latent: {latent_spatialglue.shape}")
else:
    print(f"SpatialGlue latent not found: {spatialglue_path}")

# Load STAMP latent
latent_stamp = None
stamp_path = BASELINES_DIR / "latent_stamp.npy"
if stamp_path.exists():
    latent_stamp = np.load(stamp_path)
    print(f"Loaded STAMP latent: {latent_stamp.shape}")
else:
    print(f"STAMP latent not found: {stamp_path}")

# Load MultiVI latent
latent_multivi = None
multivi_path = BASELINES_DIR / "latent_multivi.npy"
if multivi_path.exists():
    latent_multivi = np.load(multivi_path)
    print(f"Loaded MultiVI latent: {latent_multivi.shape}")
else:
    print(f"MultiVI latent not found: {multivi_path}")

# Load MOFA+ latent
latent_mofa = None
mofa_mdata_path = BASELINES_DIR / "mdata_mofa.h5mu"
mofa_latent_path = BASELINES_DIR / "latent_mofa.npy"
if mofa_latent_path.exists():
    latent_mofa = np.load(mofa_latent_path)
    print(f"Loaded MOFA+ latent: {latent_mofa.shape}")
elif mofa_mdata_path.exists():
    mdata_mofa = mu.read_h5mu(mofa_mdata_path)
    latent_mofa = np.asarray(mdata_mofa.obsm["X_mofa"])
    print(f"Loaded MOFA+ latent from MuData: {latent_mofa.shape}")
else:
    print(f"MOFA+ latent not found")

## 5. Compute spatial metrics for all models

In [ ]:
def compute_metrics_for_latent(latent, name, spatial_coords):
    """Compute spatial autocorrelation metrics for a latent representation."""
    if latent is None:
        return None

    print(f"Computing metrics for {name}...")
    try:
        spatial_metrics = compute_spatial_autocorrelation(latent, spatial_coords)
        topic_morans = compute_topic_spatial_coherence(latent, spatial_coords)
        return {
            "model": name,
            "morans_i": spatial_metrics["morans_i"],
            "topic_morans_i": topic_morans,
            "n_clusters": spatial_metrics["n_clusters"],
            "latent": latent,
            "leiden": spatial_metrics["leiden"],
        }
    except Exception as e:
        print(f"  Error computing metrics: {e}")
        return None


baseline_results = {}

# Compute metrics for baselines
if latent_spatialglue is not None:
    baseline_results["SpatialGlue"] = compute_metrics_for_latent(
        latent_spatialglue, "SpatialGlue", spatial_coords
    )

if latent_stamp is not None:
    baseline_results["STAMP"] = compute_metrics_for_latent(
        latent_stamp, "STAMP", spatial_coords
    )

if latent_multivi is not None:
    baseline_results["MultiVI"] = compute_metrics_for_latent(
        latent_multivi, "MultiVI", spatial_coords
    )

if latent_mofa is not None:
    baseline_results["MOFA+"] = compute_metrics_for_latent(
        latent_mofa, "MOFA+", spatial_coords
    )

# Add best topic model
if theta_best is not None:
    baseline_results["Topic Model (Best)"] = {
        "model": "Topic Model (Best)",
        "morans_i": best_result["morans_i"],
        "topic_morans_i": best_result["topic_morans_i"],
        "n_clusters": best_result["n_clusters"],
        "latent": theta_best,
        "leiden": best_result["leiden"],
    }

In [ ]:
# Summary comparison table
if baseline_results:
    summary_data = []
    for name, res in baseline_results.items():
        if res is not None:
            summary_data.append({
                "Model": res["model"],
                "Latent Moran's I": res["morans_i"],
                "Topic Moran's I": res["topic_morans_i"],
                "N Clusters": res["n_clusters"],
            })

    summary_df = pd.DataFrame(summary_data).sort_values("Topic Moran's I", ascending=False)
    print("\nSpatial Autocorrelation Comparison:")
    print(summary_df.to_string(index=False))
else:
    print("No models loaded for comparison.")
    summary_df = None

## 6. Plot performance comparison

In [ ]:
if summary_df is not None and len(summary_df) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Moran's I comparison
    ax = axes[0]
    x = np.arange(len(summary_df))
    width = 0.35
    ax.bar(x - width/2, summary_df["Latent Moran's I"], width, label="Latent Moran's I", color="steelblue")
    ax.bar(x + width/2, summary_df["Topic Moran's I"], width, label="Topic/Dim Moran's I", color="darkorange")
    ax.set_ylabel("Moran's I")
    ax.set_title("Spatial Autocorrelation Comparison")
    ax.set_xticks(x)
    ax.set_xticklabels(summary_df["Model"], rotation=45, ha="right")
    ax.legend()
    ax.axhline(y=0, color="gray", linestyle="--", alpha=0.5)

    # Number of clusters
    ax = axes[1]
    ax.bar(summary_df["Model"], summary_df["N Clusters"], color="seagreen", alpha=0.8)
    ax.set_ylabel("Number of Clusters")
    ax.set_title("Leiden Clustering (resolution=0.5)")
    ax.set_xticklabels(summary_df["Model"], rotation=45, ha="right")

    plt.tight_layout()

    # Save figure
    BASELINES_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(BASELINES_DIR / "spatial_comparison.png", dpi=200, bbox_inches="tight")
    plt.show()

    print(f"\nSaved comparison plot to: {BASELINES_DIR / 'spatial_comparison.png'}")

## 7. UMAP and Spatial plots for all models

In [ ]:
def plot_umap_and_spatial(latent, leiden, spatial_coords, title, figsize=(14, 5)):
    """
    Plot UMAP and spatial coordinates colored by Leiden clustering.
    """
    # Create AnnData
    adata = ad.AnnData(np.asarray(latent))
    adata.obsm["spatial"] = np.asarray(spatial_coords)
    adata.obs["leiden"] = pd.Categorical(leiden)

    # Compute UMAP
    sc.pp.neighbors(adata, use_rep="X", n_neighbors=15, metric="cosine")
    sc.tl.umap(adata, min_dist=0.3)

    # Plot
    fig, axes = plt.subplots(1, 2, figsize=figsize)

    # UMAP
    sc.pl.umap(adata, color="leiden", ax=axes[0], show=False, title=f"{title}: UMAP")

    # Spatial plot
    scatter = axes[1].scatter(
        spatial_coords[:, 0],
        spatial_coords[:, 1],
        c=leiden.astype(int) if not isinstance(leiden[0], str) else LabelEncoder().fit_transform(leiden),
        cmap="tab20",
        s=10,
        alpha=0.8,
    )
    axes[1].set_title(f"{title}: Spatial")
    axes[1].set_xlabel("X")
    axes[1].set_ylabel("Y")
    axes[1].set_aspect("equal")
    axes[1].invert_yaxis()  # Usually spatial coords have inverted y

    plt.tight_layout()
    return fig, adata

In [ ]:
# Plot for all models
all_figures = {}

for name, res in baseline_results.items():
    if res is not None:
        print(f"\nPlotting {name}...")
        fig, adata = plot_umap_and_spatial(
            res["latent"],
            res["leiden"],
            spatial_coords,
            name,
        )
        all_figures[name] = fig
        plt.show()

        # Save individual figures
        safe_name = name.lower().replace(" ", "_").replace("(", "").replace(")", "")
        fig.savefig(BASELINES_DIR / f"umap_spatial_{safe_name}.png", dpi=200, bbox_inches="tight")
        print(f"  Saved to: {BASELINES_DIR / f'umap_spatial_{safe_name}.png'}")

## 8. Combined comparison figure

In [ ]:
# Create a combined figure with all models
n_models = len([r for r in baseline_results.values() if r is not None])

if n_models > 0:
    fig, axes = plt.subplots(n_models, 2, figsize=(12, 4 * n_models))
    if n_models == 1:
        axes = axes.reshape(1, -1)

    for idx, (name, res) in enumerate([(k, v) for k, v in baseline_results.items() if v is not None]):
        latent = res["latent"]
        leiden = res["leiden"]

        # Create temporary AnnData for UMAP
        adata_temp = ad.AnnData(np.asarray(latent))
        adata_temp.obsm["spatial"] = np.asarray(spatial_coords)
        adata_temp.obs["leiden"] = pd.Categorical(leiden)

        sc.pp.neighbors(adata_temp, use_rep="X", n_neighbors=15, metric="cosine")
        sc.tl.umap(adata_temp, min_dist=0.3)

        # UMAP
        sc.pl.umap(adata_temp, color="leiden", ax=axes[idx, 0], show=False, title=f"{name}: UMAP")

        # Spatial
        leiden_encoded = leiden.astype(int) if not isinstance(leiden[0], str) else LabelEncoder().fit_transform(leiden)
        axes[idx, 1].scatter(
            spatial_coords[:, 0],
            spatial_coords[:, 1],
            c=leiden_encoded,
            cmap="tab20",
            s=8,
            alpha=0.8,
        )
        axes[idx, 1].set_title(f"{name}: Spatial (Moran's I={res['topic_morans_i']:.3f})")
        axes[idx, 1].set_aspect("equal")
        axes[idx, 1].invert_yaxis()
        axes[idx, 1].set_xlabel("X")
        axes[idx, 1].set_ylabel("Y")

    plt.tight_layout()
    fig.savefig(BASELINES_DIR / "combined_comparison.png", dpi=200, bbox_inches="tight")
    plt.show()

    print(f"\nSaved combined comparison to: {BASELINES_DIR / 'combined_comparison.png'}")

## 9. Save results

In [ ]:
# Save summary tables
BASELINES_DIR.mkdir(parents=True, exist_ok=True)

if summary_df is not None:
    summary_df.to_csv(BASELINES_DIR / "spatial_metrics_comparison.csv", index=False)
    print(f"Saved comparison metrics to: {BASELINES_DIR / 'spatial_metrics_comparison.csv'}")

if results:
    df_sorted.to_csv(BASELINES_DIR / "topic_model_metrics.csv", index=False)
    print(f"Saved topic model metrics to: {BASELINES_DIR / 'topic_model_metrics.csv'}")

## Notes

- This notebook expects trained topic models under `/data/omics_topic_models/mouse_brain_spatial` with `prior_*` directories.
- Baseline outputs are expected under `/data/omics_topic_models/mouse_brain_spatial/baselines`.
- The best model is selected by **Topic Moran's I**, which measures spatial autocorrelation of topic proportions.
- Higher Moran's I indicates topics capture spatial structure better.
- STAMP only uses RNA data, while SpatialGlue, MultiVI, and MOFA+ use both RNA and ATAC.

### Troubleshooting Model Loading

If topic models fail to load with architecture errors (GCN encoders), this is because spatial models 
have a different architecture than non-spatial models. Solutions:

1. **Re-run training** with the updated `train_mouse_brain_spatial.py` script which now saves 
   `latent_representation.npy` alongside each model.

2. **Manually save latent representations** for already-trained models by running inference 
   with the correct setup (see helper cell below if needed).

In [ ]:
# HELPER: Extract and save latent representations from already-trained models
# Run this cell only if models were trained but latent_representation.npy was not saved

EXTRACT_LATENTS = False  # Set to True to run extraction

if EXTRACT_LATENTS and model_dirs:
    print("Extracting latent representations from trained models...")
    print("This requires proper setup with spatial connectivities.")
    
    for model_dir in model_dirs:
        model_path = model_dir / "model"
        latent_path = model_dir / "latent_representation.npy"
        
        if not model_path.exists():
            continue
        if latent_path.exists():
            print(f"  {model_dir.name}: latent already exists, skipping")
            continue
            
        print(f"  Processing {model_dir.name}...")
        try:
            # Load model with full spatial setup
            model = MultimodalAmortizedLDA.load(str(model_path), adata=adata_flat)
            theta = model.get_latent_representation(batch_size=mdata.n_obs)
            np.save(latent_path, theta.values)
            print(f"    Saved latent to {latent_path}")
        except Exception as e:
            print(f"    Failed: {e}")
            print("    You may need to manually re-run training for this model")